In [11]:
import numpy as np
from pyscf import gto, scf, ci, cc, fci

# Specify nuclear coordinates and pick a basis set for the orbitals

In [12]:
mol = gto.M(atom='H 0 0 0; H 0 0 0.74', basis='sto-3g')

# Run Hartree Fock

In [13]:
mf = scf.RHF(mol).run()

converged SCF energy = -1.11675930739643


# Form the 1- and 2-body integrals in the Molecular Orbital basis
Usually this is not needed, PySCF handles everything in the background.

This is only for demonstration. Later we will need the integrals for the Jordan-Wigner transform.

We can contract the Hartree Fock Slater determinant to get the Hartree Fock energy!

In [14]:

# Transform integrals to Molecular Orbital basis
C = mf.mo_coeff
h1e_mo = C.T @ mf.get_hcore() @ C
eri_mo = np.einsum('pqrs,pi,qj,rk,sl->ijkl', mol.intor('int2e'), C, C, C, C)

# Number of occupied spatial orbitals
n_occ = mol.nelectron // 2
occ_h1e = h1e_mo[:n_occ, :n_occ]
occ_eri = eri_mo[:n_occ, :n_occ, :n_occ, :n_occ]

# Contract to recover RHF electronic energy
# E_HF = 2*sum_i h_ii + sum_{ij} [2*(ii|jj) - (ij|ji)]
e1e = 2.0 * np.trace(occ_h1e)
coulomb = np.einsum('iijj->', occ_eri)  # sum_{ij} (ii|jj)
exchange = np.einsum('ijij->', occ_eri) # sum_{ij} (ij|ji)
e2e = 2.0 * coulomb - exchange

e_hf_elec = e1e + e2e
e_hf_total = e_hf_elec + mol.energy_nuc()  # Add nuclear repulsion

print(f"PySCF reported HF energy:   {mf.e_tot:.8f} Ha")
print(f"Contracted HF energy:       {e_hf_total:.8f} Ha")

PySCF reported HF energy:   -1.11675931 Ha
Contracted HF energy:       -1.11675931 Ha


# Let's run some well known classical computational chemistry methods in PySCF

In [15]:
# Classical correlation methods
E_hf  = mf.e_tot
myci = ci.CISD(mf)
myci.kernel()
E_cisd = myci.e_tot
mycc = cc.CCSD(mf)
mycc.kernel()
E_ccsd = mycc.e_tot
E_ccsdt = mycc.ccsd_t() + E_ccsd
myfci = fci.FCI(mol, mf.mo_coeff)
myfci.kernel()
E_fci  = myfci.e_tot

print(f"\nHF:      {E_hf:.6f} Ha \nCISD:    {E_cisd:.6f} Ha \nCCSD:    {E_ccsd:.6f} Ha \nCCSD(T): {E_ccsdt:.6f} Ha \nFCI:     {E_fci:.6f} Ha")

E(RCISD) = -1.137283834488502  E_corr = -0.02052452709207651
E(CCSD) = -1.13728399861044  E_corr = -0.02052469121401453
CCSD(T) correction = 0

HF:      -1.116759 Ha 
CISD:    -1.137284 Ha 
CCSD:    -1.137284 Ha 
CCSD(T): -1.137284 Ha 
FCI:     -1.137284 Ha
